In [ ]:

@dataclass
class FineTuneConfig:
    """Fine-tuning configuration - inherits base Config values"""
    
    pretrained_unet_path: str = "wg2diff_roi_v3/final/unet"
    pretrained_mask_encoder_path: str = "wg2diff_roi_v3/final/mask_encoder.pt"
    
    base_model: str = "runwayml/stable-diffusion-v1-5"
    image_size: int = 512
    latent_size: int = 64
    roi_margin_frac: float = 0.30
    min_roi_pixels: int = 24
    
    use_mask_distance: bool = True
    mask_channels: int = 3  
    
    use_context_hint: bool = True
    context_blur_radius: float = 6.0
    ctx_drop_prob: float = 0.30
    
  
    in_channels_latent: int = 4
    in_channels_ctx_latent: int = 4
    in_channels_unet: int = 11 
    
    prompt: str = "a realistic clinical photograph of a leprosy ulcer"  
    cfg_drop_prob: float = 0.10
    
    lr: float = 5e-6  # Lower learning rate for fine-tuning
    weight_decay: float = 1e-2
    batch_size: int = 2
    grad_accum: int = 4
    num_workers: int = 4
    max_steps: int = 5000  # Fewer steps for fine-tuning
    mixed_precision: bool = True
    
    # Min-SNR weighting
    use_min_snr: bool = True
    min_snr_gamma: float = 5.0
    
    # EMA
    use_ema: bool = True
    ema_decay: float = 0.9999
    continue_ema: bool = True  
    
    # Sampling
    infer_steps: int = 30
    guidance_scale: float = 5.0
    seed: int = 42
    
    output_dir: str = "wg2diff_roi_v3_finetuned"
    save_ema_weights: bool = True
    
    freeze_unet_encoder: bool = True  # Freeze UNet encoder blocks
    freeze_mask_encoder: bool = False  # Keep mask encoder trainable

ft_cfg = FineTuneConfig()

if ft_cfg.use_mask_distance and _HAS_SCIPY:
    ft_cfg.mask_channels = 3  # binary, dist_in, dist_out
else:
    ft_cfg.mask_channels = 1  # binary only

ft_cfg.in_channels_unet = (
    ft_cfg.in_channels_latent + 
    ft_cfg.mask_channels + 
    (ft_cfg.in_channels_ctx_latent if ft_cfg.use_context_hint else 0)
)

print(f" Fine-tune config initialized:")
print(f"   mask_channels: {ft_cfg.mask_channels}")
print(f"   in_channels_unet: {ft_cfg.in_channels_unet}")
print(f"   Prompt: '{ft_cfg.prompt}'")

In [ ]:

@torch.no_grad()
def _encode_text(model, prompt: str) -> torch.Tensor:
    tokens = model.tokenizer(
        prompt,
        padding="max_length",
        truncation=True,
        max_length=model.tokenizer.model_max_length,
        return_tensors="pt",
    )
    return model.text_encoder(tokens.input_ids.to(model.device)).last_hidden_state

@torch.no_grad()
def _encode_image_to_latents(model, pixel_values: torch.Tensor) -> torch.Tensor:
    posterior = model.vae.encode(pixel_values).latent_dist
    latents = posterior.sample() * model.latent_scaling
    return latents

@torch.no_grad()
def _decode_latents_to_image(model, latents: torch.Tensor) -> torch.Tensor:
    latents = latents / model.latent_scaling
    img = model.vae.decode(latents).sample
    img = (img / 2 + 0.5).clamp(0, 1)
    return img

def _maskfeats_to_latent(model, mask_feats_512: torch.Tensor) -> torch.Tensor:
    return F.interpolate(mask_feats_512, size=(model.cfg.latent_size, model.cfg.latent_size), mode="nearest")
    

def expand_down_residuals_to_sd15(down4):
    """
    Match residuals to SD1.5 architecture by spatial size and channels.
    SD1.5 expects: [320@64, 640@32, 1280@16, 1280@8] x3 each = 12 total
    """
    assert len(down4) == 4, f"Expected 4 residuals, got {len(down4)}"
    
    res_by_size = {}
    for res in down4:
        spatial_size = res.shape[-1]
        res_by_size[spatial_size] = res
    
    # SD1.5 architecture order: 64, 32, 16, 8
    expected_order = [64, 32, 16, 8]
    expected_channels = [320, 640, 1280, 1280]
    
    sorted_residuals = []
    for size, expected_ch in zip(expected_order, expected_channels):
        if size not in res_by_size:
            raise ValueError(f"Missing residual for spatial size {size}x{size}")
        
        res = res_by_size[size]
        actual_ch = res.shape[1]
        
        if actual_ch != expected_ch:
            raise ValueError(
                f"Channel mismatch at {size}x{size}: "
                f"got {actual_ch} channels, expected {expected_ch}. "
                f"Check MaskEncoderLite output channels!"
            )
        
        sorted_residuals.append(res)
    
    out = []
    for res in sorted_residuals:
        out.extend([res, res, res])
    
    return out
def debug_residuals(down_residuals, label=""):
    """Print shapes of all residuals for debugging"""
    print(f"\n{label} Residuals ({len(down_residuals)} total):")
    for i, r in enumerate(down_residuals):
        print(f"  [{i}]: {r.shape}")

def load_pretrained_model(cfg: FineTuneConfig, device: torch.device):
    """
    Load the pre-trained WG2DiffROI_V3 model with trained weights.
    """
    print(" Loading pre-trained model")
    
    model = nn.Module()
    model.cfg = cfg
    model.device = device
    
    print(" Loading VAE and text encoder")
    model.vae = AutoencoderKL.from_pretrained(cfg.base_model, subfolder="vae").to(device)
    model.text_encoder = CLIPTextModel.from_pretrained(cfg.base_model, subfolder="text_encoder").to(device)
    model.tokenizer = CLIPTokenizer.from_pretrained(cfg.base_model, subfolder="tokenizer")
    

    print(" Initializing UNet architecture")
    model.unet = UNet2DConditionModel.from_pretrained(cfg.base_model, subfolder="unet").to(device)
    

    print(f" Expanding conv_in: 4 → {cfg.in_channels_unet} channels.")
    old = model.unet.conv_in
    new_conv = nn.Conv2d(
        in_channels=cfg.in_channels_unet,
        out_channels=old.out_channels,
        kernel_size=old.kernel_size,
        stride=old.stride,
        padding=old.padding,
    ).to(device)
    
    with torch.no_grad():
        new_conv.weight.zero_()
        new_conv.bias.copy_(old.bias)
        new_conv.weight[:, :cfg.in_channels_latent].copy_(old.weight)
        if cfg.in_channels_unet > cfg.in_channels_latent:
            nn.init.normal_(new_conv.weight[:, cfg.in_channels_latent:], mean=0.0, std=1e-3)
    
    model.unet.conv_in = new_conv
    
    # Load TRAINED weights
    if os.path.exists(cfg.pretrained_unet_path):        
        checkpoint_path = os.path.join(cfg.pretrained_unet_path, "diffusion_pytorch_model.safetensors")
        if not os.path.exists(checkpoint_path):
            checkpoint_path = os.path.join(cfg.pretrained_unet_path, "diffusion_pytorch_model.bin")
    else:
        raise FileNotFoundError(f"UNet checkpoint directory not found: {cfg.pretrained_unet_path}")
    
    model.train_scheduler = DDPMScheduler.from_pretrained(cfg.base_model, subfolder="scheduler")
    
    model.mask_encoder = MaskEncoderLite(c_mask=cfg.mask_channels).to(device)
    if os.path.exists(cfg.pretrained_mask_encoder_path):
        print(f"📂 Loading MaskEncoder from: {cfg.pretrained_mask_encoder_path}")
        mask_state = torch.load(cfg.pretrained_mask_encoder_path, map_location=device)
        model.mask_encoder.load_state_dict(mask_state)
        print(f"    Loaded {len(mask_state)} mask encoder parameters")
        del mask_state
    else:
        raise FileNotFoundError(f"MaskEncoder checkpoint not found: {cfg.pretrained_mask_encoder_path}")
    
    model.vae.requires_grad_(False)
    model.text_encoder.requires_grad_(False)
    
    model.latent_scaling = 0.18215
    
    model.encode_text = lambda prompt: _encode_text(model, prompt)
    model.encode_image_to_latents = lambda pixel_values: _encode_image_to_latents(model, pixel_values)
    model.decode_latents_to_image = lambda latents: _decode_latents_to_image(model, latents)
    model.maskfeats_to_latent = lambda mask_feats_512: _maskfeats_to_latent(model, mask_feats_512)
 
    return model


In [ ]:

def freeze_model_parts(model, cfg: FineTuneConfig):
    """
    Optionally freeze parts of the model for efficient fine-tuning.
    """
    if cfg.freeze_unet_encoder:
        print("❄️ Freezing UNet down blocks...")
        for name, param in model.unet.named_parameters():
            if "down_blocks" in name:
                param.requires_grad = False
    
    if cfg.freeze_mask_encoder:
        print("❄️ Freezing MaskEncoder...")
        for param in model.mask_encoder.parameters():
            param.requires_grad = False
    
    unet_trainable = sum(p.numel() for p in model.unet.parameters() if p.requires_grad)
    mask_trainable = sum(p.numel() for p in model.mask_encoder.parameters() if p.requires_grad)
    unet_total = sum(p.numel() for p in model.unet.parameters())
    mask_total = sum(p.numel() for p in model.mask_encoder.parameters())
    
    print(f"🔢 UNet trainable: {unet_trainable:,} / {unet_total:,} ({100*unet_trainable/unet_total:.1f}%)")
    print(f"🔢 MaskEncoder trainable: {mask_trainable:,} / {mask_total:,} ({100*mask_trainable/mask_total:.1f}%)")

In [ ]:
def finetune_v3_from_model(
    model,  
    image_paths: list,
    mask_paths: list,
    cfg: FineTuneConfig,
    device: torch.device,
):
   
    set_seed(cfg.seed)
    os.makedirs(cfg.output_dir, exist_ok=True)
    sample_dir = os.path.join(cfg.output_dir, "samples")
    os.makedirs(sample_dir, exist_ok=True)
    
    model.to(device)
    
    freeze_model_parts(model, cfg)
    
    model.train()
    
    dataset = ROIWoundDatasetV3(image_paths, mask_paths, cfg)
    loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        collate_fn=collate_skip_v3,
        drop_last=True,
    )
    
    params = [p for p in model.unet.parameters() if p.requires_grad]
    params += [p for p in model.mask_encoder.parameters() if p.requires_grad]
    
    optimizer = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda" and cfg.mixed_precision))
    
    ema = None
    if cfg.use_ema:
        ema = EMA(model.unet, decay=cfg.ema_decay)
    
    with torch.no_grad():
        text_cond = model.encode_text(cfg.prompt)
        text_uncond = model.encode_text("")
    
    global_step = 0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(total=cfg.max_steps, desc="Fine-tuning from model", dynamic_ncols=True)
    
    while global_step < cfg.max_steps:
        for batch in loader:
            if batch is None:
                continue
            
            pixel_values = batch["pixel_values"].to(device, non_blocking=True)
            ctx_pixels = batch["ctx_pixels"].to(device, non_blocking=True)
            mask_feats_512 = batch["mask_feats"].to(device, non_blocking=True)
            bsz = pixel_values.shape[0]
            
            # Mask to latent
            mask_lat = model.maskfeats_to_latent(mask_feats_512)
            
            # Encode images
            with torch.no_grad():
                latents = model.encode_image_to_latents(pixel_values)
            
            # Context hint
            if cfg.use_context_hint:
                with torch.no_grad():
                    ctx_latents = model.encode_image_to_latents(ctx_pixels)
                drop = (torch.rand(bsz, device=device) < cfg.ctx_drop_prob).float().view(bsz,1,1,1)
                ctx_latents = ctx_latents * (1.0 - drop)
            else:
                ctx_latents = None
            
            # CFG dropout
            use_cond = (torch.rand(bsz, device=device) > cfg.cfg_drop_prob)
            enc = torch.where(
                use_cond[:, None, None],
                text_cond.repeat(bsz, 1, 1),
                text_uncond.repeat(bsz, 1, 1),
            )
            
            # Timesteps
            timesteps = torch.randint(
                0, model.train_scheduler.config.num_train_timesteps,
                (bsz,), device=device, dtype=torch.long
            )
            
            # Add noise
            noise = torch.randn_like(latents)
            noisy_latents = model.train_scheduler.add_noise(latents, noise, timesteps)
            
            # UNet input
            if cfg.use_context_hint:
                unet_in = torch.cat([noisy_latents, mask_lat, ctx_latents], dim=1)
            else:
                unet_in = torch.cat([noisy_latents, mask_lat], dim=1)
            
           
            # Forward pass WITH residuals
            with torch.amp.autocast(
                device_type='cuda',
                enabled=(cfg.mixed_precision and device.type == "cuda"),
                dtype=torch.float16
            ):
                noise_pred = model.unet(
                    unet_in,
                    timesteps,
                    encoder_hidden_states=enc,
                ).sample
            
            # Loss
            mse = (noise_pred - noise) ** 2
            mse = mse.mean(dim=(1,2,3))
            
            if cfg.use_min_snr:
                w = compute_snr_weights(model.train_scheduler, timesteps, cfg.min_snr_gamma)
                loss = (w * mse).mean()
            else:
                loss = mse.mean()
            
            # Backprop
            loss = loss / cfg.grad_accum
            scaler.scale(loss).backward()
            
            if (global_step + 1) % cfg.grad_accum == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                if ema is not None:
                    ema.update(model.unet)
            
            global_step += 1
            pbar.update(1)
            pbar.set_postfix({"loss": float(loss.detach().cpu())})
            
            if global_step >= cfg.max_steps:
                break
        
        if global_step >= cfg.max_steps:
            break
    
    pbar.close()
    
    # Save fine-tuned model
    final_dir = os.path.join(cfg.output_dir, "final")
    os.makedirs(final_dir, exist_ok=True)
    
    unet_dir = os.path.join(final_dir, "unet")
    os.makedirs(unet_dir, exist_ok=True)
    
    if ema is not None and cfg.save_ema_weights:
        ema.store(model.unet)
        ema.copy_to(model.unet)
        model.unet.save_pretrained(unet_dir, safe_serialization=True)
        torch.save(ema.shadow, os.path.join(final_dir, "ema_shadow.pt"))
        ema.restore(model.unet)
    else:
        model.unet.save_pretrained(unet_dir, safe_serialization=True)
    
    torch.save(model.mask_encoder.state_dict(), os.path.join(final_dir, "mask_encoder.pt"))
    
    with open(os.path.join(cfg.output_dir, "finetune_config.txt"), "w") as f:
        for k, v in cfg.__dict__.items():
            f.write(f"{k}: {v}\n")
    
    print(f"Saved to: {cfg.output_dir}")

    return model, ema

In [ ]:
#  Run Fine-Tuning 

FINETUNE_IMAGE_DIRS = [ "/home/sagemaker-user/cleaned_images/cleaned_images"]
FINETUNE_MASK_DIRS = [ "/home/sagemaker-user/final_dataset/final_dataset/resized_leprosy/masks"]

ft_image_paths, ft_mask_paths = collect_pairs(FINETUNE_IMAGE_DIRS, FINETUNE_MASK_DIRS)

print(f"📊 Fine-tuning dataset size: {len(ft_image_paths)} pairs")

if len(ft_image_paths) == 0:
    print("No fine-tuning pairs")
else:
    ft_ds = ROIWoundDatasetV3(ft_image_paths, ft_mask_paths, ft_cfg)
    sample = None
    for i in range(min(50, len(ft_ds))):
        s = ft_ds[i]
        if not s.get("skip", False):
            sample = s
            break
    
    
ft_model, ft_ema = finetune_v3_from_model(
    model, 
    ft_image_paths,
    ft_mask_paths,
    ft_cfg,
    device
)
